In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
ROOT = Path.cwd().paren

## Tabular

In [ ]:
df = pd.read_csv(f'{ROOT}/data/isic-2024-challenge/train-metadata.csv')
df = df.ffill()
display(df.head(2))
print(df.shape)

/var/folders/z7/gxr_p2dj2mjg29pcwf45qlq40000gn/T/ipykernel_30685/2185188052.py:5: DtypeWarning: Columns (51,52) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(f'{ROOT}/data/isic-2024-challenge/train-metadata.csv')


,isic_id,target,patient_id,age_approx,sex,anatom_site_general,clin_size_long_diam_mm,image_type,tbp_tile_type,tbp_lv_A,...,lesion_id,iddx_full,iddx_1,iddx_2,iddx_3,iddx_4,iddx_5,mel_mitotic_index,mel_thick_mm,tbp_lv_dnn_lesion_confidence
0,ISIC_0015670,0,IP_1235828,60.0,male,lower extremity,3.04,TBP tile: close-up,3D: white,20.244422,...,NaN,Benign,Benign,NaN,NaN,NaN,NaN,NaN,NaN,97.517282
1,ISIC_0015845,0,IP_8170065,60.0,male,head/neck,1.10,TBP tile: close-up,3D: white,31.712570,...,IL_6727506,Benign,Benign,NaN,NaN,NaN,NaN,NaN,NaN,3.141455


(401059, 55)


split the dataset into train validation test

In [6]:
from sklearn.model_selection import StratifiedGroupKFold

df = df.reset_index(drop=True)

TEST_SPLIT = 0.1
N_SPLITS = int(1 / TEST_SPLIT)
RANDOM_STATE = 42

sgkf = StratifiedGroupKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)
train_idx, test_idx = next(sgkf.split(X=np.zeros(len(df)), y=df['target'], groups=df['patient_id']))

train_df = df.iloc[train_idx].copy()
test_df = df.iloc[test_idx].copy()

In [7]:
assert set(train_df["patient_id"]).isdisjoint(set(test_df["patient_id"]))  # no group overlap
print("Train size:", len(train_df), "| Test size:", len(test_df))
print("Class dist (train):")
print(train_df["target"].value_counts(normalize=True))
print("Class dist (test):")
print(test_df["target"].value_counts(normalize=True))

Train size: 363483 | Test size: 37576
Class dist (train):
target
0    0.998999
1    0.001001
Name: proportion, dtype: float64
Class dist (test):
target
0    0.999228
1    0.000772
Name: proportion, dtype: float64


In [8]:
path = f"{ROOT}/data/new/"

train_df.to_csv(path + "train-metadata.csv", index=False)

test_df.to_csv(path + "test-metadata.csv", index=False)


## Images

In [9]:
path = f"{ROOT}/data/new/"

train_df = pd.read_csv(path + "train-metadata.csv")
test_df = pd.read_csv(path + "test-metadata.csv")

/var/folders/z7/gxr_p2dj2mjg29pcwf45qlq40000gn/T/ipykernel_30685/2727820117.py:3: DtypeWarning: Columns (51,52) have mixed types. Specify dtype option on import or set low_memory=False.
  train_df = pd.read_csv(path + "train-metadata.csv")


In [10]:
import matplotlib.pyplot as plt
from PIL import Image

img = Image.open('/Users/cgiovannoni/Projects/isic24-dum/data/isic-2024-challenge/train-image/image/ISIC_0015670.jpg')
img.show()
print(img.size)


(139, 139)


In [17]:
from pathlib import Path

def create_new_img_dataset(img_folder, patient_id):
    new_map = []

    for file in img_folder.glob('*.jpg'):
        if file.stem in patient_id:
            new_map.append(file)
            
    return new_map


IMG_FOLDER = Path(f"{ROOT}/data/isic-2024-challenge/train-image/image")

patient_ids_test = set(test_df['isic_id'].to_list())
patient_ids_train = set(train_df['isic_id'].to_list())

train_images = create_new_img_dataset(IMG_FOLDER, patient_ids_test)
# create_new_img_dataset(IMG_FOLDER, patient_ids_train)

In [16]:
train_images




[PosixPath('/Users/cgiovannoni/Projects/isic24-dum/data/isic-2024-challenge/train-image/image/ISIC_0249489.jpg'),
 PosixPath('/Users/cgiovannoni/Projects/isic24-dum/data/isic-2024-challenge/train-image/image/ISIC_4366837.jpg'),
 PosixPath('/Users/cgiovannoni/Projects/isic24-dum/data/isic-2024-challenge/train-image/image/ISIC_4121508.jpg'),
 PosixPath('/Users/cgiovannoni/Projects/isic24-dum/data/isic-2024-challenge/train-image/image/ISIC_4626996.jpg'),
 PosixPath('/Users/cgiovannoni/Projects/isic24-dum/data/isic-2024-challenge/train-image/image/ISIC_4931617.jpg'),
 PosixPath('/Users/cgiovannoni/Projects/isic24-dum/data/isic-2024-challenge/train-image/image/ISIC_9115889.jpg'),
 PosixPath('/Users/cgiovannoni/Projects/isic24-dum/data/isic-2024-challenge/train-image/image/ISIC_9537161.jpg'),
 PosixPath('/Users/cgiovannoni/Projects/isic24-dum/data/isic-2024-challenge/train-image/image/ISIC_6248419.jpg'),
 PosixPath('/Users/cgiovannoni/Projects/isic24-dum/data/isic-2024-challenge/train-image/

In [ ]:
mapping_